# LanceDB - a vector database for LLM application

In [1]:
import lancedb

db = lancedb.connect(uri = "vector_database")
db

LanceDBConnection(uri='c:\\Users\\lisay\\Documents\\8.Ai_ops\\ai_ops_course\\09_lancedb\\vector_database')

In [2]:
db.uri

'c:\\Users\\lisay\\Documents\\8.Ai_ops\\ai_ops_course\\09_lancedb\\vector_database'

## Create a table

In [3]:
import json 

with open("data/animals_text_embeddings.json", "r") as file:
    data = json.loads(file.read()) 

data

[{'text': 'A small brown dog running.', 'vector': [0.12, 0.85, 0.33]},
 {'text': 'A cat resting quietly on a sofa.', 'vector': [0.4, 0.91, 0.1]},
 {'text': 'A large gray elephant drinking water.',
  'vector': [0.88, 0.22, 0.55]},
 {'text': 'A fast cheetah sprinting across the savannah.',
  'vector': [0.95, 0.12, 0.72]},
 {'text': 'A colorful parrot perched on a branch.',
  'vector': [0.25, 0.66, 0.81]},
 {'text': 'A frog sitting on a lily pad.', 'vector': [0.14, 0.44, 0.27]}]

In [4]:
table_animals = db.create_table("animals_text", exist_ok=True, data = data)
table_animals

LanceTable(name='animals_text', _conn=LanceDBConnection(uri='c:\\Users\\lisay\\Documents\\8.Ai_ops\\ai_ops_course\\09_lancedb\\vector_database'))

In [5]:
table_animals = db.create_table("animals_text", exist_ok=True, data = data, mode = "overwrite")
table_animals

LanceTable(name='animals_text', _conn=LanceDBConnection(uri='c:\\Users\\lisay\\Documents\\8.Ai_ops\\ai_ops_course\\09_lancedb\\vector_database'))

In [6]:
table_animals.head()

pyarrow.Table
text: string
vector: fixed_size_list<item: float>[3]
  child 0, item: float
----
text: [["A small brown dog running.","A cat resting quietly on a sofa.","A large gray elephant drinking water.","A fast cheetah sprinting across the savannah.","A colorful parrot perched on a branch."]]
vector: [[[0.12,0.85,0.33],[0.4,0.91,0.1],[0.88,0.22,0.55],[0.95,0.12,0.72],[0.25,0.66,0.81]]]

In [7]:
table_animals.to_pandas()

,text,vector
0,A small brown dog running.,"[0.12, 0.85, 0.33]"
1,A cat resting quietly on a sofa.,"[0.4, 0.91, 0.1]"
2,A large gray elephant drinking water.,"[0.88, 0.22, 0.55]"
3,A fast cheetah sprinting across the savannah.,"[0.95, 0.12, 0.72]"
4,A colorful parrot perched on a branch.,"[0.25, 0.66, 0.81]"
5,A frog sitting on a lily pad.,"[0.14, 0.44, 0.27]"


In [8]:
more_data = [
    {"text": "A panda eating bamboo pacefully", "vector": [0.51, 0.37, 0.82]},
    {"text": "A lion roaring loudly on a rock", "vector": [0.93, 0.18, 0.41]},
]

table_animals.add(more_data)

AddResult(version=21)

In [9]:
table_animals.to_pandas()

,text,vector
0,A small brown dog running.,"[0.12, 0.85, 0.33]"
1,A cat resting quietly on a sofa.,"[0.4, 0.91, 0.1]"
2,A large gray elephant drinking water.,"[0.88, 0.22, 0.55]"
3,A fast cheetah sprinting across the savannah.,"[0.95, 0.12, 0.72]"
4,A colorful parrot perched on a branch.,"[0.25, 0.66, 0.81]"
5,A frog sitting on a lily pad.,"[0.14, 0.44, 0.27]"
6,A panda eating bamboo pacefully,"[0.51, 0.37, 0.82]"
7,A lion roaring loudly on a rock,"[0.93, 0.18, 0.41]"


## Create an empty table and delete it

In [14]:
from lancedb.pydantic import LanceModel

# A Pydantic Model base class that can be converted into a LanceDB Table 
class JokeSchema(LanceModel):
    joke: str
    rating: int

db.create_table(name= "jokes", schema= JokeSchema, exist_ok=True)

db

ValueError: Schema Error: Provided schema does not match existing table schema

In [ ]:
db.table_names()

C:\Users\lisay\AppData\Local\Temp\ipykernel_28712\763631344.py:1: DeprecationWarning: table_names() is deprecated, use list_tables() instead
  db.table_names()


['animals_text', 'jokes']

In [ ]:
db.drop_table("jokes")

In [ ]:
db.table_names()

C:\Users\lisay\AppData\Local\Temp\ipykernel_28712\763631344.py:1: DeprecationWarning: table_names() is deprecated, use list_tables() instead
  db.table_names()


['animals_text']

## Open existing table

In [ ]:
db.open_table("animals_text").head(2)

pyarrow.Table
text: string
vector: fixed_size_list<item: float>[3]
  child 0, item: float
----
text: [["A small brown dog running.","A cat resting quietly on a sofa."]]
vector: [[[0.12,0.85,0.33],[0.4,0.91,0.1]]]

## Vector search in LanceDB

In [ ]:
table_animals.to_pandas()

,text,vector
0,A small brown dog running.,"[0.12, 0.85, 0.33]"
1,A cat resting quietly on a sofa.,"[0.4, 0.91, 0.1]"
2,A large gray elephant drinking water.,"[0.88, 0.22, 0.55]"
3,A fast cheetah sprinting across the savannah.,"[0.95, 0.12, 0.72]"
4,A colorful parrot perched on a branch.,"[0.25, 0.66, 0.81]"
5,A frog sitting on a lily pad.,"[0.14, 0.44, 0.27]"
6,A panda eating bamboo pacefully,"[0.51, 0.37, 0.82]"
7,A lion roaring loudly on a rock,"[0.93, 0.18, 0.41]"


In [ ]:
query_vector = [0.5, 0.2, 0.9]

table_animals.search(query_vector).limit(3).to_pandas()

,text,vector,_distance
0,A panda eating bamboo pacefully,"[0.51, 0.37, 0.82]",0.0354
1,A fast cheetah sprinting across the savannah.,"[0.95, 0.12, 0.72]",0.2413
2,A large gray elephant drinking water.,"[0.88, 0.22, 0.55]",0.2673


## Embedding API
- idea: want to put in text -> and it will automatacally generate vector embeddings
- put in a query and it will automatacally generate vector embeddings 
- calculate closest distance

LanceDB provides an Embeddings API that allows you to easily generate and store vector embeddings for your data. This is particularly useful for applications such as semantic search, recommendation systems, and clustering.

We'll use embeddings model from sentence-transformer here

In [ ]:
from lancedb.pydantic import LanceModel, Vector
from lancedb.embeddings import get_registry

model = get_registry().get(
    "sentence-transformers").create(name="paraphrase-multilingual-MiniLM-L12-v2")

class JokeModel(LanceModel):
    joke: str = model.SourceField() # marks text as input to embedding function
    vector: Vector(model.ndims()) = model.VectorField() # vector is destination of computed embedding

table_jokes = db.create_table("jokes", schema = JokeModel, exist_ok=True)
table_jokes

c:\Users\lisay\Documents\8.Ai_ops\ai_ops_course\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\lisay\Documents\8.Ai_ops\ai_ops_course\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\lisay\.cache\huggingface\hub\models--sentence-transformers--paraphrase-multilingual-MiniLM-L12-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Deve

ValueError: Schema Error: Provided schema does not match existing table schema

In [12]:
import pandas as pd 
with open("data/jokes.json", "r", encoding="utf-8") as file:
    jokes_data = json.loads(file.read())

df_jokes = pd.DataFrame(jokes_data).rename({"jokes": "joke"}, axis=1)
df_jokes.head()

,joke
0,Parallel lines have so much in common—it’s sad...
1,"ETL stands for “Extract, Transform, Leave for ..."
2,What do you call a snake that runs your script...
3,"Gold walks into a bar. The bartender says, “Au..."
4,C# devs don’t argue; they just throw exceptions.


In [13]:
table_jokes.add(df_jokes)

NameError: name 'table_jokes' is not defined